# Figs. 4–5 — Spatial SNN

In [ ]:
# Setup
# Patient FIT/CAL timings are observations; every model trajectory below is newly integrated.
import _setup
import numpy as np
from modeling import computational as snn, panel_analysis as panels
from modeling.experiments import search_core_counts, simulate_selected, angle_response
from analysis import from_events as analysis, spectral_fields as spectral, model_transition as transition
from plotting import fig4, fig5, video
from plotting.computational import energy_alignment
from workflows.tutorial import model_inputs, TutorialOutput, save_model_analysis
get_ipython().run_line_magic('matplotlib', 'inline')
config, patient = model_inputs()
out = TutorialOutput('snn')

In [ ]:
# Model definition
# Delayed LIF neurons: 40,000 cells, E:I = 4:1, a 20 x 20 mm sheet, dt = 0.1 ms.
# The paper resamples only E-to-E edges with perpendicular width x1.5.
# Core thresholds, OU input, connection degree, weights and delays retain their original rules.
substrate = snn.build_substrate(config)

In [ ]:
# Fitting objective
# FIT defines rank, within/inter-shaft timing, and participation distributions.
# Independent CAL subsamples scale the three discrepancies; J is their equal-weight mean.
objective = snn.fit_objective(patient.fit, patient.cal, patient.names,
    n_fourier=config['n_fourier'], cal_draw_indices=patient.cal_draw_indices)
# Use Timing+Space patient templates; freeze the FIT rank classifier for model modes.
patient_templates = panels.fit_patient_templates(patient.fit, patient.names, patient.coords_mm)

In [ ]:
# Fig. 4A: Model, local sampling and a newly generated event
# Use the paper's local E/I circuit painter, with the new sheet and readout arrays.
baseline = snn.simulate(substrate, config['centers_mm'], patient.contacts, seed=config['training_seed'])
snn.score_simulation(baseline, patient.observer, objective, config['burnin_ms'])
out.show(fig4.model_definition(baseline, config['radius_mm']), 'fig4_A_model')
out.simulation('baseline_new_simulation', baseline)

In [ ]:
# Core optimization
# Independent Sobol starts, followed by the paper's Matern-5/2 GP, qLogEI and uncertainty proposals.
# Fewer than eight scorable observations uses the original space-filling fallback.
# All core counts receive equal budgets. Demo: 10 evaluations x 2 restarts; paper: 31 x 4.
# Each evaluation integrates a new network; invalid trials consume budget without a fabricated loss.
searches = search_core_counts(substrate, patient.contacts, patient.observer, objective,
    out.path/'core_search', core_counts=(1, 2, 3, 4),
    budget=config['optimization_budget'], restarts=config['restarts'])
rows = [row for search in searches for row in search['rows']]

In [ ]:
# Fig. 4B: Three observable errors
# Each line is one newly evaluated configuration; color denotes evaluation order.
# These descriptive errors are distinct from the calibrated training loss J.
out.show(fig4.errors(rows), 'fig4_B_observable_errors')

In [ ]:
# Fig. 4C: Core-count comparison
# Call the original painter: mean best-so-far loss +/- sample SD across restarts.
# Use only epochs shared by every restart; an unscorable trial cannot become a numeric loss.
out.show(fig4.optimization_curves(searches), 'fig4_C_core_search')

In [ ]:
# Fig. 4D: E-to-E angle response
# Run all 13 paper scan angles, -20 to +20 degrees, with fixed remaining parameters and noise.
# Rotate the connection kernel and rebuild its E-to-E edges before every simulation.
angle_rows = angle_response(config, config['centers_mm'], patient.contacts, patient.observer,
    objective, out.path/'angle_scan')
out.show(fig4.errors(angle_rows, angle=True), 'fig4_D_direction_response')

In [ ]:
# Fig. 4E: Core-position density
# Take the lowest-J 20% of the newly evaluated two-core configurations.
# Keep the original 0.35-mm Gaussian density, 90%-mass contours and electrode display.
# These are search locations, not a posterior distribution.
out.show(fig4.core_density(rows, patient.contacts, patient.names), 'fig4_E_core_positions')

In [ ]:
# Select the fitted network
# Select the two-core configuration by training J, then rerun it from its parameters and seed.
# The score must agree with its search evaluation. Subsequent panels use this fitted run.
fitted_substrate, fitted, selected = simulate_selected(config, searches, patient.contacts,
    patient.observer, objective)
out.simulation('selected_fit', fitted)

In [ ]:
# Fig. 4F: Continuous contact waveforms
# Filter the full spike-density readout at 30-80 Hz before selecting the continuous window.
# Use one common q99 amplitude across contacts; shade the two selected recruitment spans.
from plotting.fig4_waveform import waveforms
out.show(waveforms(fitted, patient_templates, patient.names), 'fig4_F_fitted_waveforms')

In [ ]:
# Fig. 4G: Patient and model ranks
# Keep the paper's SCL/ICL contact order; model profiles are solid, patient profiles dashed.
matching = panels.model_patient_comparison(fitted['primary_centroids_ms'], patient.fit,
    patient.names, patient.contact_contract, patient_templates, n_null=1000)
out.show(fig4.rank_profiles(matching, patient.names), 'fig4_G_mean_rank')

In [ ]:
# Fig. 4H: Held-out contact matching
# Assign templates on one contact half and score on the other; swap halves and combine.
# Recompute the within-shaft permutation test before drawing significance annotations.
out.show(fig4.crossfit_matrix(matching), 'fig4_H_crossfit')

In [ ]:
# Fig. 4I: Template similarity
# Call the paper's TA-MTA / TB-MTB similarity painter for the E1146 demo patient.
# The paper's 25-patient distribution requires independent fits for those patients.
# A different-noise simulation is not a substitute for a patient.
out.show(fig4.template_similarity(matching), 'fig4_I_template_similarity')

In [ ]:
# Supplementary Video 2
# Compare the new baseline and fitted network; order rows by their measured training error.
# Select a chronological event of each template class and align each to its own first centroid.
# Keep paired envelope/field columns, a 0-1 envelope scale, and a 0-75 native activity scale.
comparison, selection = panels.select_propagation_events([baseline, fitted],
    patient_templates, patient.names, config['burnin_ms'])
video.selected_propagation_movie(comparison, selection, out.figures/'supplementary_video2_new.gif', config['radius_mm'])

In [ ]:
# Fig. 5A: Z/M simulation and contact waveforms
# Follow the six-panel reference_layout_20261011 mapping, which remains a separate figure version.
# Keep the Fig. 4 fitted network fixed; enable Z/M and sample current at native 10 kHz.
# Allow up to 60 s for the newly fitted network; stop at max(15 s, native high-rate entry + 3 s).
# The paper painter filters 80-1000 Hz, then applies fixed pre-recruitment P99.5 scaling.
zm = snn.simulate(fitted_substrate, fitted['centers'], patient.contacts, outward=fitted['outward'],
    seed=config['training_seed'], zm=config['zm'], record_current=True, duration_ms=60000.)
onset = transition.display_entry(zm)
entry = spectral.native_entry(zm['time_ms'], zm['rate_E'], dt_ms=config['parameters']['dt'])
context = fig5.prepare(zm, config['zm']['eta_m'], onset)
out.show(fig5.waveform(context), 'fig5_A_waveforms')
out.simulation('zm_enabled', zm)

In [ ]:
# Fig. 5B: Slow variables
# Draw All-E (solid) and Core-A (dashed) Z and effective adaptation eta*M from this simulation.
out.show(fig5.slow_variables(context), 'fig5_B_slow_variables')

In [ ]:
# Fig. 5C: Spatial recruitment
# Count actual E spikes in 20-ms windows and divide by the local E-cell count and duration.
# Use the paper's magma/PowerNorm display, fixed at 0-500 Hz.
out.show(fig5.spatial_activity(context), 'fig5_C_spatial_activity')

In [ ]:
# Fig. 5D: State trajectory
# The selected six-panel version plots Z, eta*M, and E rate; the older version used inhibition.
# Call the original trajectory painter, with 5-ms rate bins and normalized simulation time.
out.show(fig5.state_trajectory(context), 'fig5_D_state_trajectory')

In [ ]:
# Fig. 5E: Model and clinical onset fields
# Recompute CAR, 1-150 Hz power and baseline robust-z from new currents and raw clinical voltages.
# Fix the shared plane from interictal observations before projecting either onset field.
# The patient illustration uses positive mean enhancement, then the highest signed field correlation.
from analysis import clinical_fields as clinical
from plotting import clinical_fields as clinical_plot
from workflows.clinical import prepare_clinical
energy = spectral.energy_field_trajectory(zm['contact_current'], 10000., patient.coords_mm,
    patient_templates['templates'][0], patient_templates['support'][0], baseline_sec=(1.5, 3.), band=(30, 80))
onset_field = transition.onset_broadband_field(zm['contact_current'], 10000., onset)
clinical_rows = prepare_clinical(patient.names)
comparison = clinical.descriptive_example(onset_field, clinical_rows, energy['scorer']['weight_id'])
out.show(clinical_plot.model_and_patient(onset_field, comparison, energy), 'fig5_E_model_patient_fields')
# The paper displays min-max spatial shape; keep actual model z values in a separate view.
out.show(fig5.onset_field(zm, onset_field, energy, normalized=False), 'supplementary_onset_signed_z')

In [ ]:
# Fig. 5F: Selection-corrected field statistic
# Use the paper's dB readout without MAD division; this differs from panel E's robust-z example.
# Detect the final model recruitment and quiet baseline from the new trajectory.
# Restrict the 200-ms power peak to final-ramp start - 100 ms through native entry + 600 ms.
# Clinical windows follow the six-band phase rule, recomputed from raw recordings above.
# Permute all model contacts before smoothing and repeat best-seizure selection in every draw.
# One E1146 patient contributes one observed/null pair; the same aggregator accepts a patient cohort.
model_energy = clinical.model_contrast(zm)
patient_statistic = clinical.patient_statistic(model_energy, clinical_rows, energy['scorer']['weight_id'])
statistic = clinical.cohort_statistic([patient_statistic])
out.show(clinical_plot.statistic(statistic), 'fig5_F_field_statistic')

In [ ]:
# Supplement: Gamma-field scores and saved calculations
# Compute the energy field's correlation/projection onto the fixed interictal field.
from workflows.clinical import save_model_analysis
out.show(energy_alignment(energy), 'supplementary_gamma_field_scores')
save_model_analysis(out, config, objective, matching, selection, onset, entry, onset_field, energy,
    angle_rows=angle_rows, clinical_rows=clinical_rows, model_energy=model_energy,
    comparison=comparison, statistic=statistic, patient_statistic=patient_statistic)